In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from pyspark.sql.functions import *
from pyspark.sql.types import *

##### Importing from the custom utility package

In [0]:
from utils.custom_utils import transformations
obj = transformations()

#### Reading bronze table

In [0]:
df = spark.table("lakehouse.bronze.cust_info")

### Silver Layer Transformations

#### 1. Renaming Columns

In [0]:
rename_config = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "create_date"
}

for old_name, new_name in rename_config.items():
    df = df.withColumnRenamed(old_name, new_name) 

#### 2. Trimming

In [0]:
df = obj.trim_columns(df)

#### 3. Removing records with no customer_id

In [0]:
df = obj.remove_null(df = df, primary_col = "customer_id")

#### 4. Removing customer_id duplicates
- Since create_date behaves as last_updated in our data source, hence it can be used as CDC(Change Data Capture)

In [0]:
df = obj.dedup(df = df, dedup_cols = ["customer_id"], cdc = "create_date")

#### 5. Normalization

In [0]:
df = (
    df.withColumn(
        "marital_status",
        when(upper(col("marital_status")) == "M", "Married")
        .when(upper(col("marital_status")) == "S", "Single")
        .otherwise("n/a")
    )
    .withColumn(
        "gender",
        when(upper(col("gender")) == "M", "Male")
        .when(upper(col("gender")) == "F", "Female")
        .otherwise("n/a")
    )
)

#### 6. Adding ingestion_ts column

In [0]:
df = obj.add_ingestion_timestamp(df)

#### DataFrame Sanity Check

In [0]:
df.limit(20).display()

#### 7. Applying upsert(SCD type 1)

In [0]:
target_table = "lakehouse.silver.crm_customers"

if spark.catalog.tableExists(target_table):
    obj.upsert(spark = spark, df = df, key_cols = ["customer_id"], table = "crm_cutomers",
                cdc = "create_date")
else:
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

#### Silver table sanity check

In [0]:
%sql
select *
from lakehouse.silver.crm_customers
limit 20